In [2]:
!pip install -q gradio

In [4]:
import gradio as gr
import chromadb
from sentence_transformers import SentenceTransformer
from pathlib import Path
import os
import sys
import json
sys.path.append(str(Path.cwd().parent / "src"))

In [5]:
try:
    from rag_pipeline import ComplaintRAG
    print("✅ Imported ComplaintRAG from src/rag_pipeline.py")
except ImportError:
    # Fallback: define the class inline (in case they don't have the script)
    print("⚠️ Could not import from rag_pipeline.py. Defining class inline.")
    
    class ComplaintRAG:
        def __init__(self, vector_store_path=None, embedding_model="all-MiniLM-L6-v2"):
            import chromadb
            from sentence_transformers import SentenceTransformer
            from pathlib import Path
            
            if vector_store_path is None:
                PROJECT_ROOT = Path.cwd().parent
                vector_store_path = PROJECT_ROOT / "vector_store" / "chroma_db"
            
            self.model = SentenceTransformer(embedding_model)
            self.client = chromadb.PersistentClient(path=str(vector_store_path))
            self.collection = self.client.get_collection("complaints")
            self.top_k = 5
            print(f"✅ Loaded vector store with {self.collection.count()} chunks")
        
        def retrieve(self, query, top_k=None):
            if top_k is None:
                top_k = self.top_k
            query_embedding = self.model.encode([query])[0]
            results = self.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            documents = results['documents'][0] if results['documents'] else []
            metadatas = results['metadatas'][0] if results['metadatas'] else []
            distances = results['distances'][0] if results['distances'] else []
            retrieved = []
            for doc, meta, dist in zip(documents, metadatas, distances):
                retrieved.append({
                    'text': doc,
                    'complaint_id': meta.get('complaint_id', 'Unknown'),
                    'product': meta.get('product', 'Unknown'),
                    'score': 1 - dist
                })
            return retrieved
        
        def generate_answer(self, query, retrieved_chunks, llm_function):
            context_parts = []
            for i, chunk in enumerate(retrieved_chunks):
                context_parts.append(f"[Source {i+1}] Product: {chunk['product']}\n{chunk['text']}")
            context = "\n\n---\n\n".join(context_parts)
            prompt = f"""You are a financial analyst assistant for CreditTrust. Your task is to answer questions about customer complaints. Use the following retrieved complaint excerpts to formulate your answer. If the context doesn't contain the answer, state that you don't have enough information.

Context:
{context}

Question: {query}

Answer:"""
            return llm_function(prompt)
        
        def query(self, query, llm_function, top_k=None):
            retrieved = self.retrieve(query, top_k)
            answer = self.generate_answer(query, retrieved, llm_function)
            return {'question': query, 'answer': answer, 'sources': retrieved}

✅ Imported ComplaintRAG from src/rag_pipeline.py


In [8]:
# Cell 4: Define the LLM function (mock for now)

def mock_llm(prompt):
    return f"[Mock Answer] The system retrieved relevant complaint excerpts. For real answers, please set up a real LLM."

# --- Optional: Hugging Face API (uncomment to use) ---
# import os
# os.environ["HF_TOKEN"] = "your_token_here"  # Set your token
# 
# def use_huggingface_api(prompt, model="mistralai/Mistral-7B-Instruct-v0.3"):
#     import requests
#     token = os.getenv("HF_TOKEN")
#     if not token:
#         return mock_llm(prompt)
#     headers = {"Authorization": f"Bearer {token}"}
#     response = requests.post(
#         f"https://api-inference.huggingface.co/models/{model}",
#         headers=headers,
#         json={"inputs": prompt}
#     )
#     if response.status_code == 200:
#         return response.json()[0]['generated_text']
#     return mock_llm(prompt)

# Choose which LLM function to use
llm_function = mock_llm
print("✅ Using mock LLM (replace with real LLM for production)")

✅ Using mock LLM (replace with real LLM for production)


In [9]:
# Cell 5: Initialize RAG system

rag = ComplaintRAG()
print("✅ RAG system ready.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Loaded vector store with 5852 chunks
✅ RAG system ready.


In [11]:
# Cell 6: Define the Gradio interface

def respond(question):
    """Function that takes a question and returns answer + sources"""
    if not question.strip():
        return "Please enter a question.", ""
    
    # Query the RAG system
    result = rag.query(question, llm_function)
    answer = result['answer']
    sources = result['sources']
    
    # Format sources for display
    sources_text = ""
    if sources:
        for i, src in enumerate(sources, 1):
            sources_text += f"\n**[Source {i}]** Product: {src['product']} (Score: {src['score']:.3f})\n{src['text'][:300]}...\n\n"
    else:
        sources_text = "No relevant sources found."
    
    return answer, sources_text

# Create Gradio Blocks
with gr.Blocks(title="CreditTrust Complaint Analyst") as demo:
    gr.Markdown("""
    # CreditTrust Complaint Analyst
    Ask questions about customer complaints across financial products.
    The system retrieves relevant complaint excerpts and generates an answer.
    """)
    
    with gr.Row():
        with gr.Column(scale=2):
            question_input = gr.Textbox(
                label="Your Question",
                placeholder="e.g., Why are people unhappy with Credit Cards?",
                lines=2
            )
            submit_btn = gr.Button("Ask", variant="primary")
            clear_btn = gr.Button("Clear")
        with gr.Column(scale=1):
            pass  # for spacing
    
    answer_output = gr.Textbox(
        label="Answer",
        lines=6,
        interactive=False
    )
    sources_output = gr.Textbox(
        label="Retrieved Sources",
        lines=8,
        interactive=False,
        placeholder="Sources will appear here..."
    )
    
    # Event handlers
    submit_btn.click(
        respond,
        inputs=[question_input],          # only the question input
        outputs=[answer_output, sources_output]
    )
    question_input.submit(
        respond,
        inputs=[question_input],
        outputs=[answer_output, sources_output]
    )
    clear_btn.click(
        lambda: ("", ""),                # clear both input and sources output
        outputs=[question_input, sources_output]
    )

print("✅ Gradio interface defined. Run cell 7 to launch.")

✅ Gradio interface defined. Run cell 7 to launch.


In [12]:
# Cell 7: Launch the app

demo.launch(share=False)  # set share=True for a public link

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


C:\Users\user\OneDrive\Desktop\Project\KAIM\credittrust-complaint-rag\credittrust-complaint-rag\venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
